# 03 — Demand analysis

Volume/volatility segments, intermittency (ADI / CV²), rolling demand, YoY when history allows. These segments drive later inventory comments.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.logging_config import setup_logging
from src.utils.helpers import load_config, load_model_config, resolve_path
setup_logging("INFO")
CONFIG = load_config()
print("Independent M5-schema project. DATA_DIR =", resolve_path(CONFIG["paths"]["data_dir"]))


In [ ]:
import pandas as pd
from src.features.demand_features import build_demand_profile

processed = resolve_path(CONFIG["paths"]["processed_dir"]) / "fact_daily_sales.parquet"
if not processed.exists():
    raise SystemExit("Run python scripts/run_pipeline.py first")
fact = pd.read_parquet(processed)
profile = build_demand_profile(fact)
print(profile["demand_segment"].value_counts())
print(profile["demand_pattern"].value_counts())
profile.sort_values("coefficient_of_variation", ascending=False).head(10)


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
profile.boxplot(column="coefficient_of_variation", by="cat_id", ax=ax)
ax.set_title("Demand CV by category")
ax.set_ylabel("CV")
plt.suptitle("")
plt.show()
